# Straw defect thesis — reproducible experiments
Attach the private `straw_thesis_source.zip` Dataset and the original straw image Dataset. Enable GPU. Internet is needed if pretrained ImageNet weights are not cached. Preserve the full output between sessions.
Core = 27 training runs; extended = 48. Pilot a single group before estimating total time. This notebook creates new results; it does not certify earlier result tables. Existing images have already been used in development.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, subprocess, shutil
WORK = Path('/kaggle/working/straw_thesis')
SOURCE = Path('/kaggle/working/straw_thesis_source')
archives = sorted(Path('/kaggle/input').rglob('straw_thesis_source.zip'))
assert len(archives) == 1, 'Attach exactly one source ZIP dataset.'
SOURCE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        assert (SOURCE / member.filename).resolve().is_relative_to(SOURCE.resolve()), 'Unsafe ZIP path'
    archive.extractall(SOURCE)
os.chdir(SOURCE)
sys.path.insert(0, str(SOURCE))
roots = sorted(p for p in Path('/kaggle/input').rglob('Straw Defects in Juice Box') if p.is_dir())
assert len(roots) == 1, 'Set DATA_ROOT manually to the folder containing all brand folders.'
DATA_ROOT = roots[0]
print('Source:', SOURCE, 'Data:', DATA_ROOT)


In [ ]:
# Retain Kaggle's matching torch/torchvision CUDA installations.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements/straw-thesis.txt'], check=True)
import torch, torchvision
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Notebook settings.'
print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'pip-freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_straw_thesis.py', '-v'], check=True)


In [ ]:
MODULE = 'src.straw_defect_cls.thesis_runner'
PROTOCOL = WORK / 'data/iid'
if not (PROTOCOL / 'protocol.json').exists():
    subprocess.run([sys.executable, '-m', MODULE, 'prepare', '--data-root', str(DATA_ROOT), '--output', str(WORK / 'data')], check=True)
print(json.loads((PROTOCOL / 'protocol.json').read_text())['counts'])


In [ ]:
SUITE = 'core'  # extended includes core and all 21 additional IID runs
RESULT = WORK / f'{SUITE}_v1'
PHASES = ['plan', 'train', 'calibrate', 'freeze', 'test', 'benchmark', 'report']
# For a pilot/shard set PHASES=['plan','train'] and RUN_IDS to the three IDs below.
# RUN_IDS = [f'core_resnet18_r224_s42_stage{s}' for s in ['1','2','full5']]
RUN_IDS = []
for phase in PHASES:
    command = [sys.executable, '-u', '-m', MODULE, phase, '--protocol', str(PROTOCOL), '--output', str(RESULT), '--suite', SUITE]
    if phase == 'train':
        for run_id in RUN_IDS:
            command.extend(['--run-id', run_id])
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)


In [ ]:
# Optional: all eight held-out-brand experiments (24 additional training runs).
RUN_LOBO = False
if RUN_LOBO:
    subprocess.run([sys.executable, '-m', MODULE, 'lobo-data', '--protocol', str(PROTOCOL), '--output', str(WORK / 'lobo_data')], check=True)
    for fold in sorted((WORK / 'lobo_data').iterdir()):
        for phase in ['train', 'calibrate', 'freeze', 'test', 'report']:
            subprocess.run([sys.executable, '-u', '-m', MODULE, phase, '--protocol', str(fold), '--output', str(WORK / 'lobo_results' / fold.name)], check=True)
    subprocess.run([sys.executable, 'scripts/research/aggregate_straw_lobo.py', '--root', str(WORK / 'lobo_results'), '--output', str(WORK / 'lobo_summary')], check=True)


In [ ]:
# Download all outputs, source bundle and environment. Do not export only best.pt.
shutil.copy2(archives[0], WORK / 'source_bundle.zip')
from IPython.display import display, HTML
report = RESULT / 'report/index.html'
if report.exists():
    print('Report:', report)
    display(HTML(report.read_text()))
print('Preserve /kaggle/working/straw_thesis using Save Version with outputs; large ZIP creation is optional.')
